# 🧠 TB & MS Prediction — Model Training Walkthrough

This notebook walks through the complete training process.

**Steps:**
1. Setup & Imports
2. Build Data Generators
3. Build Model (EfficientNetB0)
4. Phase 1 Training (frozen base)
5. Phase 2 Fine-tuning
6. Evaluation on Test Set
7. Grad-CAM Visualization

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

sys.path.insert(0, os.path.dirname(os.getcwd()))

from config import *
from src.preprocessing import build_data_generators, get_class_weights, plot_sample_images
from src.model import build_model, unfreeze_for_finetuning, print_model_summary
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_roc_curve

print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs Available: {len(gpus)}')

# Set disease to train
DISEASE = 'tb'  # Change to 'ms' for MS model
print(f'\n🔬 Training: {DISEASE_CONFIG[DISEASE]["name"]}')

## Step 1: Build Data Generators

In [ ]:
train_gen, val_gen, test_gen = build_data_generators(DISEASE, batch_size=BATCH_SIZE)

# Class weights for imbalanced data
class_weights = get_class_weights(train_gen)
print(f'Class Weights: {class_weights}')

In [ ]:
# Visualise sample images
plot_sample_images(train_gen, DISEASE)

## Step 2: Build Model

In [ ]:
model = build_model(learning_rate=LEARNING_RATE_PHASE1, trainable_base=False)
print_model_summary(model)

## Step 3: Phase 1 Training (Frozen Base)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks_p1 = [
    EarlyStopping(monitor='val_auc', mode='max', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                      min_lr=1e-7, verbose=1),
]

print('🚀 Phase 1: Training custom head (base frozen)...')
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE1,
    class_weight=class_weights,
    callbacks=callbacks_p1,
    verbose=1
)

## Step 4: Phase 2 Fine-Tuning

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

model = unfreeze_for_finetuning(model, n_layers=FINE_TUNE_LAYERS,
                                 learning_rate=LEARNING_RATE_PHASE2)

callbacks_p2 = [
    ModelCheckpoint(
        DISEASE_CONFIG[DISEASE]['model_path'],
        monitor='val_auc', mode='max',
        save_best_only=True, verbose=1
    ),
    EarlyStopping(monitor='val_auc', mode='max', patience=PATIENCE_EARLY_STOP,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=PATIENCE_REDUCE_LR,
                      min_lr=MIN_LR, verbose=1),
]

print('🔬 Phase 2: Fine-tuning last layers...')
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE2,
    class_weight=class_weights,
    callbacks=callbacks_p2,
    verbose=1
)

## Step 5: Training History

In [ ]:
from src.train import plot_training_history
plot_training_history(history1, history2, DISEASE)

## Step 6: Test Set Evaluation

In [ ]:
from src.evaluate import run_evaluation
metrics = run_evaluation(DISEASE)

## Step 7: Grad-CAM Visualization

In [ ]:
from src.gradcam import visualize_gradcam

# Find a test image
test_dir = os.path.join(DISEASE_CONFIG[DISEASE]['data_dir'], 'test')
cls = 'tb' if DISEASE == 'tb' else 'ms'
cls_dir = os.path.join(test_dir, cls)

if os.path.exists(cls_dir) and os.listdir(cls_dir):
    test_img = os.path.join(cls_dir, os.listdir(cls_dir)[0])
    print(f'Running Grad-CAM on: {test_img}')
    visualize_gradcam(test_img, model, DISEASE)
else:
    print('No test images found to visualize.')

## ✅ Training Complete!

Your model is saved in `models/`. Launch the web app:
```bash
streamlit run app/streamlit_app.py
```

Or use the API:
```bash
uvicorn api.fastapi_app:app --reload
```